[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/084.%20The%20Multi-Facility%20Location%20-%20p-Center%20Problem/P84-Tier-8_executed.ipynb)

---


# 84. The Multi-Facility Location: p-Center Problem

## Tier 8: A Value-Aligned Framework

### Purpose, inputs, and outputs

This notebook studies a small facility-location example and shows how demographic equity requirements can be added to a **p-center** model. The input consists of demand-point coordinates, candidate-facility coordinates, a required number of facilities, demographic-group memberships and weights, facility categories, and ethical parameters. The outputs are a standard p-center solution, a value-aligned solution when one is feasible, group-level service-distance measures, and illustrative stakeholder and monitoring analyses.

The solution method is exhaustive enumeration: it evaluates every possible set of exactly $p$ facilities, first using the standard p-center objective and then using the value-aligned objective subject to the stated constraints. This preserves a transparent, exact solution process for the small instance in this notebook.

### Model assumptions and scope

- Distances are Euclidean distances between the supplied coordinates. The coordinates are treated as kilometres, so model outputs are distances in kilometres, **not response times in minutes**. Converting distance to response time would require a travel-speed or travel-time model, which is not included.
- Demand points and candidate facility locations are fixed, and each selected facility serves the demand point nearest to it. Capacity, road networks, traffic, facility operating costs, and uncertainty are outside this example.
- Each demographic group is represented by its listed demand points. Group service is measured by that group's maximum distance to its nearest selected facility.
- The example selects $p=4$ facilities from four candidates. Consequently, there is only one possible facility set. This instance can illustrate metrics and constraint checks, but it cannot demonstrate a change in facility locations or a meaningful location trade-off.
- The stakeholder scores and long-term monitoring scenario later in the notebook are illustrative calculations based on explicitly stated assumptions. They are not measured community preferences, observed trends, or forecasts.

### Ethical framework: concepts and sequence

The model represents equity through group-specific maximum service distances and a weighted average of those distances. A fairness constraint limits each group's maximum distance relative to that weighted average. A second constraint requires a minimum fraction of selected facilities to be in locations categorized as underserved. The objective combines the weighted equity measure with the standard p-center maximum distance.

The workflow is:

1. Identify demand groups and their weights.
2. Define the fairness and underserved-facility constraints.
3. Enumerate facility sets and evaluate their feasibility and objective values.
4. Explain the selected set using group-level metrics and constraint checks.
5. Examine how the stated parameters and illustrative stakeholder preferences affect reported metrics.

These mathematical constraints operationalize only the definitions given here; they do not, by themselves, establish that a policy is ethically sufficient or legally compliant.

### Mathematical model

Let $I$ be the set of demand points, $J$ the set of candidate facilities, and $G$ the set of demographic groups. Let $I_g \subseteq I$ be the demand points in group $g$. For each $i \in I$ and $j \in J$, let $d_{ij}$ be the Euclidean distance from demand point $i$ to candidate facility $j$. A facility set $S \subseteq J$ must contain exactly $p$ facilities.

For a given $S$, define the nearest-facility distance for demand point $i$ as

$$
r_i(S) = \min_{j \in S} d_{ij}.
$$

The standard p-center objective is the largest of these distances:

$$
R(S) = \max_{i \in I} r_i(S).
$$

For each group $g$, define its maximum service distance as

$$
R_g(S) = \max_{i \in I_g} r_i(S).
$$

Let $w_g$ be the group priority weight and $h_g$ its historical-disadvantage factor. The model uses the adjusted group weight $q_g = w_g(1+h_g)$ and the weighted average group distance

$$
\overline{R}(S) = \frac{\sum_{g \in G} q_g R_g(S)}{\sum_{g \in G} q_g}.
$$

The equity ratio is the largest group maximum divided by the smallest group maximum. If the smallest is zero while another is positive, the ratio is infinite; if every group maximum is zero, the ratio is defined as one.

The value-aligned objective is

$$
Z(S) = \alpha\,\overline{R}(S) + (1-\alpha)R(S),
$$

where $\alpha$ is `equity_weight`. A set is feasible when it satisfies both

$$
R_g(S) \leq (1+\beta)\overline{R}(S) \quad \text{for every } g \in G,
$$

and

$$
\sum_{j\in S} \mathbf{1}\{j\text{ is categorized as underserved}\}
\geq \lceil \gamma p \rceil.
$$

Here, $\beta$ controls the allowed deviation from the weighted average, and $\gamma$ specifies the minimum fraction of selected facilities that must be in underserved locations. The ceiling makes the required facility count an integer that meets or exceeds the specified fraction. The value-aligned problem minimizes $Z(S)$ over feasible sets; the standard problem minimizes $R(S)$ over all sets of size $p$.

The code computes $d_{ij}$ in `compute_distance_matrix`, $R(S)$ in `evaluate_solution`, $R_g(S)$ and $\overline{R}(S)$ in `evaluate_equity`, feasibility in `is_feasible`, and the two objectives in the solver methods. With $m=|J|$ candidate facilities and $n=|I|$ demand points, exhaustive enumeration considers $\binom{m}{p}$ sets. Evaluating a set requires distance lookups across the demand points and candidate facilities, so the total work grows combinatorially with the number of candidate facilities. This method is suitable here because the instance is deliberately small.

### Interpreting the results

The standard objective and the value-aligned objective are different quantities, so their numeric values should not be compared as if they measured the same criterion. The trade-off analysis therefore compares the two solutions' **maximum service distances**. A negative percentage means the value-aligned solution has a smaller maximum distance than the standard solution; a positive percentage means it has a larger one. The equity ratio is a descriptive measure, not a complete measure of fairness. Ties may occur; exhaustive enumeration returns one best set according to the order in which combinations are examined.

In [ ]:
from dataclasses import dataclass
from itertools import combinations
from math import ceil, isclose, isinf
from typing import Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class DemographicGroup:
    """Demand-point group and its equity-weighting characteristics."""

    group_id: int
    name: str
    demand_points: List[int]
    priority_weight: float
    historical_disadvantage: float


@dataclass
class EthicalConstraints:
    """Parameters defining the fairness and location-category constraints."""

    beta: float
    gamma: float
    equity_weight: float
    transparency_threshold: float

    def __post_init__(self) -> None:
        if not 0 <= self.beta <= 1:
            raise ValueError("beta must be between 0 and 1.")
        if not 0 <= self.gamma <= 1:
            raise ValueError("gamma must be between 0 and 1.")
        if not 0 <= self.equity_weight <= 1:
            raise ValueError("equity_weight must be between 0 and 1.")
        if not 0 <= self.transparency_threshold <= 1:
            raise ValueError("transparency_threshold must be between 0 and 1.")


@dataclass
class PCenterInstance:
    """A p-center instance with group membership and facility categories."""

    demand_points: List[Tuple[float, float]]
    facility_locations: List[Tuple[float, float]]
    p: int
    demographic_groups: List[DemographicGroup]
    facility_categories: List[str]

    def __post_init__(self) -> None:
        if not self.demand_points:
            raise ValueError("At least one demand point is required.")
        if not self.facility_locations:
            raise ValueError("At least one candidate facility is required.")
        if not 1 <= self.p <= len(self.facility_locations):
            raise ValueError("p must be between 1 and the number of candidates.")
        if len(self.facility_categories) != len(self.facility_locations):
            raise ValueError("Each candidate facility needs one category.")
        if not self.demographic_groups:
            raise ValueError("At least one demographic group is required.")

        covered_points = []
        for group in self.demographic_groups:
            if not group.demand_points:
                raise ValueError(f"Group {group.name!r} must contain demand points.")
            if group.priority_weight < 0:
                raise ValueError("Group priority weights must be non-negative.")
            if not 0 <= group.historical_disadvantage <= 1:
                raise ValueError("Historical-disadvantage factors must be in [0, 1].")
            if any(index < 0 or index >= len(self.demand_points)
                   for index in group.demand_points):
                raise ValueError(f"Group {group.name!r} has an invalid demand index.")
            covered_points.extend(group.demand_points)

        if sorted(covered_points) != list(range(len(self.demand_points))):
            raise ValueError(
                "Groups must partition the demand points without omissions or duplicates."
            )
        if sum(
            group.priority_weight * (1 + group.historical_disadvantage)
            for group in self.demographic_groups
        ) <= 0:
            raise ValueError("The total adjusted group weight must be positive.")

    def compute_distance_matrix(self) -> np.ndarray:
        """Return Euclidean distances from every demand point to every candidate."""
        demand = np.asarray(self.demand_points, dtype=float)
        facilities = np.asarray(self.facility_locations, dtype=float)
        differences = demand[:, np.newaxis, :] - facilities[np.newaxis, :, :]
        return np.sqrt(np.sum(differences ** 2, axis=2))

    def evaluate_solution(self, facility_set: Set[int]) -> float:
        """Return the p-center maximum nearest-facility distance."""
        if not facility_set:
            raise ValueError("A solution must contain at least one facility.")
        distances = self.compute_distance_matrix()
        nearest_distances = distances[:, sorted(facility_set)].min(axis=1)
        return float(nearest_distances.max())

    def evaluate_equity(self, facility_set: Set[int]) -> Dict:
        """Return group maximum distances, their ratio, and weighted average."""
        if not facility_set:
            raise ValueError("A solution must contain at least one facility.")
        distances = self.compute_distance_matrix()
        selected = sorted(facility_set)
        group_max_distances = {}

        for group in self.demographic_groups:
            group_distances = distances[group.demand_points][:, selected].min(axis=1)
            group_max_distances[group.name] = float(group_distances.max())

        maximum = max(group_max_distances.values())
        minimum = min(group_max_distances.values())
        if isclose(minimum, 0.0, abs_tol=1e-12):
            equity_ratio = 1.0 if isclose(maximum, 0.0, abs_tol=1e-12) else float("inf")
        else:
            equity_ratio = maximum / minimum

        weighted_sum = 0.0
        total_weight = 0.0
        for group in self.demographic_groups:
            weight = group.priority_weight * (1 + group.historical_disadvantage)
            weighted_sum += group_max_distances[group.name] * weight
            total_weight += weight

        return {
            "group_max_distances": group_max_distances,
            "equity_ratio": equity_ratio,
            "weighted_average": weighted_sum / total_weight,
        }

    def check_underserved_constraint(self, facility_set: Set[int], gamma: float) -> bool:
        """Check the minimum underserved-facility count implied by gamma."""
        required_facilities = ceil(gamma * self.p - 1e-12)
        underserved_facilities = sum(
            self.facility_categories[index] == "underserved"
            for index in facility_set
        )
        return underserved_facilities >= required_facilities

In [ ]:
class ValueAlignedPCenterSolver:
    """Exhaustive-enumeration solver for standard and value-aligned p-center models."""

    def __init__(self, instance: PCenterInstance, constraints: EthicalConstraints):
        self.instance = instance
        self.constraints = constraints
        self.distances = instance.compute_distance_matrix()
        self.solution_history = []

    def compute_value_aligned_objective(self, facility_set: Set[int]) -> float:
        """Combine weighted group distance and standard p-center distance."""
        efficiency_objective = self.instance.evaluate_solution(facility_set)
        equity_metrics = self.instance.evaluate_equity(facility_set)
        equity_objective = equity_metrics["weighted_average"]
        alpha = self.constraints.equity_weight
        return alpha * equity_objective + (1 - alpha) * efficiency_objective

    def is_feasible(self, facility_set: Set[int]) -> Tuple[bool, List[str]]:
        """Check set size, the group fairness bound, and underserved count."""
        violations = []
        candidate_count = len(self.instance.facility_locations)

        if len(facility_set) != self.instance.p:
            violations.append(f"Exactly {self.instance.p} facilities must be selected.")
        if any(index < 0 or index >= candidate_count for index in facility_set):
            violations.append("The set contains an invalid facility index.")
        if violations:
            return False, violations

        equity_metrics = self.instance.evaluate_equity(facility_set)
        average_service = equity_metrics["weighted_average"]
        max_allowed = (1 + self.constraints.beta) * average_service

        for group_name, group_max in equity_metrics["group_max_distances"].items():
            if group_max > max_allowed and not isclose(
                group_max, max_allowed, rel_tol=1e-9, abs_tol=1e-12
            ):
                violations.append(
                    f"Group {group_name} exceeds the fairness bound: "
                    f"{group_max:.3f} > {max_allowed:.3f} distance units."
                )

        if not self.instance.check_underserved_constraint(
            facility_set, self.constraints.gamma
        ):
            required = ceil(self.constraints.gamma * self.instance.p - 1e-12)
            violations.append(
                f"At least {required} selected facilities must be in underserved locations."
            )

        return not violations, violations

    def solve_standard(self) -> Tuple[Set[int], float, Dict]:
        """Minimize the standard p-center maximum distance by enumeration."""
        best_solution = None
        best_objective = float("inf")

        for combo in combinations(
            range(len(self.instance.facility_locations)), self.instance.p
        ):
            facility_set = set(combo)
            objective = self.instance.evaluate_solution(facility_set)
            if objective < best_objective and not isclose(
                objective, best_objective, rel_tol=1e-12, abs_tol=1e-12
            ):
                best_objective = objective
                best_solution = facility_set

        if best_solution is None:
            raise RuntimeError("No facility set was generated for the standard problem.")
        return (
            best_solution,
            best_objective,
            self.instance.evaluate_equity(best_solution),
        )

    def solve_value_aligned(self) -> Tuple[Optional[Set[int]], float, Optional[Dict], List[str]]:
        """Minimize the combined objective over feasible sets by enumeration."""
        best_solution = None
        best_objective = float("inf")
        best_violations = []
        first_infeasible_violations = []

        for combo in combinations(
            range(len(self.instance.facility_locations)), self.instance.p
        ):
            facility_set = set(combo)
            feasible, violations = self.is_feasible(facility_set)

            if not feasible:
                if not first_infeasible_violations:
                    first_infeasible_violations = violations
                continue

            objective = self.compute_value_aligned_objective(facility_set)
            if objective < best_objective and not isclose(
                objective, best_objective, rel_tol=1e-12, abs_tol=1e-12
            ):
                best_objective = objective
                best_solution = facility_set
                best_violations = []

        if best_solution is None:
            message = ["No facility set satisfies all value-aligned constraints."]
            message.extend(first_infeasible_violations)
            return None, float("inf"), None, message

        return (
            best_solution,
            best_objective,
            self.instance.evaluate_equity(best_solution),
            best_violations,
        )

    def generate_explanation(self, facility_set: Set[int]) -> Dict:
        """Summarize selected facilities, group metrics, and constraint checks."""
        equity_metrics = self.instance.evaluate_equity(facility_set)
        feasible, violations = self.is_feasible(facility_set)
        stakeholder_impact = {}

        for group in self.instance.demographic_groups:
            group_max = equity_metrics["group_max_distances"][group.name]
            stakeholder_impact[group.name] = {
                "maximum_service_distance": group_max,
                "priority_level": group.priority_weight,
                "historical_disadvantage": group.historical_disadvantage,
                "disadvantage_flag": group.historical_disadvantage > 0.5,
            }

        return {
            "selected_facilities": sorted(facility_set),
            "overall_maximum_distance": self.instance.evaluate_solution(facility_set),
            "equity_metrics": equity_metrics,
            "constraint_satisfaction": feasible,
            "violations": violations,
            "stakeholder_impact": stakeholder_impact,
        }

    def compare_solutions(self) -> Dict:
        """Compare standard and value-aligned solutions using consistent metrics."""
        standard_solution, standard_objective, standard_equity = self.solve_standard()
        value_solution, value_objective, value_equity, violations = (
            self.solve_value_aligned()
        )

        standard_maximum = standard_objective
        if value_solution is None:
            value_maximum = None
            efficiency_tradeoff = None
            equity_improvement = None
        else:
            value_maximum = self.instance.evaluate_solution(value_solution)
            if isclose(standard_maximum, 0.0, abs_tol=1e-12):
                efficiency_tradeoff = (
                    0.0 if isclose(value_maximum, 0.0, abs_tol=1e-12) else float("inf")
                )
            else:
                efficiency_tradeoff = (
                    (value_maximum - standard_maximum) / standard_maximum * 100
                )

            standard_ratio = standard_equity["equity_ratio"]
            value_ratio = value_equity["equity_ratio"]
            if isinf(standard_ratio) and isinf(value_ratio):
                equity_improvement = 0.0
            elif isinf(standard_ratio):
                equity_improvement = 100.0
            elif isinf(value_ratio):
                equity_improvement = float("-inf")
            elif isclose(standard_ratio, 0.0, abs_tol=1e-12):
                equity_improvement = 0.0
            else:
                equity_improvement = (
                    (standard_ratio - value_ratio) / standard_ratio * 100
                )

        return {
            "standard_solution": {
                "facilities": sorted(standard_solution),
                "objective": standard_objective,
                "equity_metrics": standard_equity,
            },
            "value_aligned_solution": {
                "facilities": sorted(value_solution) if value_solution is not None else None,
                "objective": value_objective,
                "maximum_distance": value_maximum,
                "equity_metrics": value_equity,
                "violations": violations,
            },
            "trade_offs": {
                "efficiency_cost_percent": efficiency_tradeoff,
                "equity_improvement_percent": equity_improvement,
                "is_feasible": value_solution is not None,
            },
        }

## 1. Define the city-planning instance

The coordinates below define eight demand points, four candidate fire-station locations, and two demographic groups. The facility categories determine which candidates count toward the underserved-location requirement. The example sets $p=4$, so every candidate is selected; that choice is retained to make the example's data explicit, but it means facility selection itself has no alternatives in this instance.

In [ ]:
print("Value-Aligned Framework for the p-Center Problem")
print("=" * 52)

# Coordinates are measured in kilometres; distances are not travel times.
demand_points = [
    (1, 1), (2, 3), (3, 1), (4, 4),
    (8, 1), (9, 3), (10, 2), (11, 4),
]

facility_locations = [
    (0, 2), (2, 2),
    (5, 2), (12, 2),
]

facility_categories = ["underserved", "underserved", "mixed", "mixed"]
p = 4

demographic_groups = [
    DemographicGroup(
        group_id=1,
        name="Low-Income",
        demand_points=[0, 1, 2, 3],
        priority_weight=1.5,
        historical_disadvantage=0.8,
    ),
    DemographicGroup(
        group_id=2,
        name="Affluent",
        demand_points=[4, 5, 6, 7],
        priority_weight=1.0,
        historical_disadvantage=0.1,
    ),
]

instance = PCenterInstance(
    demand_points=demand_points,
    facility_locations=facility_locations,
    p=p,
    demographic_groups=demographic_groups,
    facility_categories=facility_categories,
)

constraints = EthicalConstraints(
    beta=0.3,
    gamma=0.5,
    equity_weight=0.6,
    transparency_threshold=0.8,
)

print("\nProblem setup:")
print(f"- {len(demand_points)} demand points across {len(demographic_groups)} groups")
print(f"- {len(facility_locations)} candidate facility locations")
print(f"- Select p={p} facilities")
print(f"- Coordinate-distance units: kilometres")
print(f"- Fairness tolerance beta={constraints.beta}; underserved fraction gamma={constraints.gamma}")
print("\nDemographic groups:")
for group in demographic_groups:
    print(
        f"- {group.name}: {len(group.demand_points)} demand points, "
        f"priority={group.priority_weight}, "
        f"historical disadvantage={group.historical_disadvantage}"
    )

assert len(demand_points) == 8
assert len(facility_locations) == 4
assert p == len(facility_locations)

## 2. Enumerate and compare solutions

The standard model minimizes the worst nearest-facility distance. The value-aligned model uses the weighted objective and accepts only sets satisfying both ethical constraints. The reported trade-off compares maximum distances in the same units; the combined value-aligned objective is reported separately because it has a different definition. If no set satisfies the constraints, the value-aligned result is explicitly marked infeasible rather than presenting an infeasible set as a valid solution.

In [ ]:
solver = ValueAlignedPCenterSolver(instance, constraints)
comparison = solver.compare_solutions()

standard = comparison["standard_solution"]
value_aligned = comparison["value_aligned_solution"]
trade_offs = comparison["trade_offs"]

print("Standard p-center solution:")
print(f"- Selected facilities: {standard['facilities']}")
print(f"- Maximum service distance: {standard['objective']:.3f} km")
print("- Group maximum service distances:")
for group_name, distance in standard["equity_metrics"]["group_max_distances"].items():
    print(f"  - {group_name}: {distance:.3f} km")
print(f"- Equity ratio (worst/best group maximum): {standard['equity_metrics']['equity_ratio']:.3f}")

print("\nValue-aligned solution:")
if trade_offs["is_feasible"]:
    print(f"- Selected facilities: {value_aligned['facilities']}")
    print(f"- Combined objective Z: {value_aligned['objective']:.3f} km-equivalent")
    print(f"- Maximum service distance: {value_aligned['maximum_distance']:.3f} km")
    print("- Group maximum service distances:")
    for group_name, distance in value_aligned["equity_metrics"]["group_max_distances"].items():
        print(f"  - {group_name}: {distance:.3f} km")
    print(f"- Equity ratio (worst/best group maximum): {value_aligned['equity_metrics']['equity_ratio']:.3f}")
    print("- All modeled ethical constraints are satisfied.")
else:
    print("- No facility set satisfies all modeled ethical constraints.")
    for violation in value_aligned["violations"]:
        print(f"  - {violation}")

print("\nComparison using maximum service distance:")
if trade_offs["efficiency_cost_percent"] is None:
    print("- A maximum-distance trade-off is unavailable because the value-aligned model is infeasible.")
elif isinf(trade_offs["efficiency_cost_percent"]):
    print("- The percentage trade-off is infinite because the standard maximum distance is zero.")
else:
    print(f"- Change in maximum distance: {trade_offs['efficiency_cost_percent']:+.2f}%")

if trade_offs["equity_improvement_percent"] is not None:
    print(f"- Change in equity ratio, expressed as improvement: {trade_offs['equity_improvement_percent']:+.2f}%")
print(f"- Value-aligned feasibility: {trade_offs['is_feasible']}")

# Verify that the reported standard objective agrees with its selected set.
assert isclose(
    standard["objective"],
    instance.evaluate_solution(set(standard["facilities"])),
    rel_tol=1e-12,
    abs_tol=1e-12,
)

## 3. Explain the selected set and visualize its metrics

The explanation reports what the model computes: selected candidate indices, maximum service distance, group metrics, and whether the modeled constraints hold. The plots use distinct markers and a legend to distinguish demand groups, candidate facilities, and the selected set. Because this instance selects every candidate, the standard and value-aligned selected sets coincide whenever the value-aligned constraints are feasible.

In [ ]:
if trade_offs["is_feasible"]:
    explanation = solver.generate_explanation(set(value_aligned["facilities"]))
    print("Model-based explanation:")
    print(f"- Selected stations: {[f'F{index + 1}' for index in explanation['selected_facilities']]}")
    print(f"- Overall maximum service distance: {explanation['overall_maximum_distance']:.3f} km")
    print(f"- Equity ratio: {explanation['equity_metrics']['equity_ratio']:.3f}")
    print(f"- Modeled constraints satisfied: {explanation['constraint_satisfaction']}")

    for group_name, impact in explanation["stakeholder_impact"].items():
        print(f"\n{group_name} group:")
        print(f"- Maximum service distance: {impact['maximum_service_distance']:.3f} km")
        print(f"- Priority weight: {impact['priority_level']}")
        print(f"- Historical-disadvantage factor: {impact['historical_disadvantage']:.1f}")
        print(f"- Disadvantage flag (factor > 0.5): {impact['disadvantage_flag']}")

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    ax1, ax2, ax3, ax4 = axes.flat

    group_colors = {"Low-Income": "tab:red", "Affluent": "tab:blue"}
    for group in demographic_groups:
        points = np.asarray([demand_points[i] for i in group.demand_points])
        ax1.scatter(
            points[:, 0], points[:, 1], s=90, marker="s",
            color=group_colors[group.name], label=f"{group.name} demand",
            zorder=4,
        )
        for index in group.demand_points:
            x_coord, y_coord = demand_points[index]
            ax1.annotate(f"D{index + 1}", (x_coord, y_coord), xytext=(4, 4),
                         textcoords="offset points", fontsize=8)

    for index, (x_coord, y_coord) in enumerate(facility_locations):
        ax1.scatter(
            x_coord, y_coord, s=150, marker="^", color="tab:green",
            edgecolor="black", label="Selected facility" if index == 0 else None,
            zorder=5,
        )
        ax1.annotate(f"F{index + 1}", (x_coord, y_coord), xytext=(4, -10),
                     textcoords="offset points", fontsize=8)

    ax1.set_title("Demand Points and Selected Facilities")
    ax1.set_xlabel("X coordinate (km)")
    ax1.set_ylabel("Y coordinate (km)")
    ax1.grid(True, alpha=0.3)
    ax1.axis("equal")
    ax1.legend()

    groups = list(standard["equity_metrics"]["group_max_distances"])
    standard_distances = [
        standard["equity_metrics"]["group_max_distances"][name] for name in groups
    ]
    aligned_distances = [
        value_aligned["equity_metrics"]["group_max_distances"][name] for name in groups
    ]
    positions = np.arange(len(groups))
    width = 0.36
    ax2.bar(positions - width / 2, standard_distances, width,
            label="Standard", color="tab:blue", alpha=0.8)
    ax2.bar(positions + width / 2, aligned_distances, width,
            label="Value-aligned", color="tab:orange", alpha=0.8)
    ax2.set_title("Group Maximum Service Distance")
    ax2.set_xlabel("Demographic group")
    ax2.set_ylabel("Maximum distance (km)")
    ax2.set_xticks(positions, groups)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")

    ratios = [
        standard["equity_metrics"]["equity_ratio"],
        value_aligned["equity_metrics"]["equity_ratio"],
    ]
    ratio_bars = ax3.bar(
        ["Standard", "Value-aligned"], ratios,
        color=["tab:blue", "tab:orange"], alpha=0.8,
    )
    ax3.set_title("Equity Ratio")
    ax3.set_ylabel("Worst/best group maximum")
    ax3.grid(True, alpha=0.3, axis="y")
    for bar, ratio in zip(ratio_bars, ratios):
        label = "infinite" if isinf(ratio) else f"{ratio:.3f}"
        ax3.annotate(label, (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     xytext=(0, 4), textcoords="offset points", ha="center")

    if trade_offs["efficiency_cost_percent"] is None:
        ax4.text(0.5, 0.5, "Value-aligned model is infeasible",
                 ha="center", va="center", transform=ax4.transAxes)
        ax4.set_axis_off()
    else:
        labels = ["Maximum-distance change", "Equity-ratio improvement"]
        values = [
            trade_offs["efficiency_cost_percent"],
            trade_offs["equity_improvement_percent"],
        ]
        ax4.bar(labels, values, color=["tab:blue", "tab:green"], alpha=0.8)
        ax4.axhline(0, color="black", linewidth=0.8)
        ax4.set_title("Relative Comparison")
        ax4.set_ylabel("Percentage (%)")
        ax4.tick_params(axis="x", rotation=15)
        ax4.grid(True, alpha=0.3, axis="y")

    fig.suptitle("Standard and Value-Aligned p-Center Metrics")
    fig.tight_layout()
    plt.show()
else:
    print("No value-aligned solution is available to explain or compare in the plots.")

## 4. Sensitivity to fairness and underserved-location parameters

Sensitivity analysis reruns the same exhaustive-enumeration method for each $(\beta,\gamma)$ pair. A smaller $\beta$ imposes a tighter group-distance bound. A larger $\gamma$ raises the minimum number of selected facilities that must be categorized as underserved. Results depend on the supplied instance; no parameter range is universally recommended. Since all four candidate facilities are selected here, changing $\gamma$ affects feasibility through the count requirement, not through a change in the selected locations.

In [ ]:
def analyze_parameter_sensitivity(
    instance: PCenterInstance,
    beta_values: List[float],
    gamma_values: List[float],
) -> List[Dict]:
    """Evaluate the existing value-aligned model over parameter combinations."""
    results = []
    for beta in beta_values:
        for gamma in gamma_values:
            test_constraints = EthicalConstraints(
                beta=beta,
                gamma=gamma,
                equity_weight=0.6,
                transparency_threshold=0.8,
            )
            test_solver = ValueAlignedPCenterSolver(instance, test_constraints)
            comparison = test_solver.compare_solutions()
            value_solution = comparison["value_aligned_solution"]
            trade_offs = comparison["trade_offs"]

            results.append({
                "beta": beta,
                "gamma": gamma,
                "equity_ratio": (
                    value_solution["equity_metrics"]["equity_ratio"]
                    if value_solution["equity_metrics"] is not None else np.nan
                ),
                "maximum_distance_change_percent": trade_offs["efficiency_cost_percent"],
                "is_feasible": trade_offs["is_feasible"],
            })
    return results


beta_values = [0.1, 0.2, 0.3, 0.4, 0.5]
gamma_values = [0.25, 0.5, 0.75]
sensitivity_df = pd.DataFrame(
    analyze_parameter_sensitivity(instance, beta_values, gamma_values)
)

print("Sensitivity results by beta:")
for beta in beta_values:
    subset = sensitivity_df[sensitivity_df["beta"] == beta]
    feasible_subset = subset[subset["is_feasible"]]
    if feasible_subset.empty:
        print(f"- beta={beta:.1f}: no feasible settings in the tested gamma values")
    else:
        mean_ratio = feasible_subset["equity_ratio"].mean()
        print(
            f"- beta={beta:.1f}: mean equity ratio among feasible settings "
            f"is {mean_ratio:.3f}; feasible settings "
            f"{len(feasible_subset)}/{len(subset)}"
        )

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax1, ax2, ax3, ax4 = axes.flat

for gamma in gamma_values:
    subset = sensitivity_df[sensitivity_df["gamma"] == gamma].sort_values("beta")
    ax1.plot(subset["beta"], subset["equity_ratio"], "o-", label=f"gamma={gamma:.2f}")
ax1.set_title("Equity Ratio by Fairness Tolerance")
ax1.set_xlabel("Beta: allowed deviation")
ax1.set_ylabel("Equity ratio (worst/best)")
ax1.legend(title="Underserved fraction")
ax1.grid(True, alpha=0.3)

for gamma in gamma_values:
    subset = sensitivity_df[sensitivity_df["gamma"] == gamma].sort_values("beta")
    ax2.plot(
        subset["beta"], subset["maximum_distance_change_percent"],
        "s-", label=f"gamma={gamma:.2f}"
    )
ax2.set_title("Maximum-Distance Change")
ax2.set_xlabel("Beta: allowed deviation")
ax2.set_ylabel("Change relative to standard (%)")
ax2.legend(title="Underserved fraction")
ax2.grid(True, alpha=0.3)

feasibility_matrix = sensitivity_df.pivot(
    index="gamma", columns="beta", values="is_feasible"
)
feasibility_matrix = feasibility_matrix.reindex(
    index=gamma_values, columns=beta_values
)
sns.heatmap(
    feasibility_matrix.astype(int), annot=True, fmt="d", cmap="RdYlGn",
    vmin=0, vmax=1, cbar_kws={"label": "Feasible (1) / infeasible (0)"}, ax=ax3,
)
ax3.set_title("Feasibility by Parameter Pair")
ax3.set_xlabel("Beta")
ax3.set_ylabel("Gamma")

for feasible, label, color in [
    (True, "Feasible", "tab:green"),
    (False, "Infeasible", "tab:red"),
]:
    subset = sensitivity_df[sensitivity_df["is_feasible"] == feasible]
    if not subset.empty:
        ax4.scatter(
            subset["beta"], subset["gamma"], color=color,
            label=label, s=80, alpha=0.8,
        )
ax4.set_title("Feasible Parameter Combinations")
ax4.set_xlabel("Beta")
ax4.set_ylabel("Gamma")
ax4.set_yticks(gamma_values)
ax4.legend()
ax4.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(
    "\nInterpretation: the charts describe this instance and these tested values. "
    "They do not establish a generally optimal ethical policy."
)

## 5. Illustrative stakeholder preference scoring

The following scoring exercise applies a transparent arithmetic rule to stated preference weights. It does **not** alter the p-center optimization model, and its scores are not empirical satisfaction measures. For each stakeholder, the score combines simple proxies for efficiency, equity, and the absolute maximum-distance change, then multiplies the result by that stakeholder's weight. The exercise is useful for inspecting how chosen weights affect a summary score, but it cannot establish community approval or represent community feedback.

In [ ]:
class Stakeholder:
    """Illustrative stakeholder weight and preference specification."""

    def __init__(self, name: str, weight: float, preferences: Dict[str, float]):
        self.name = name
        self.weight = weight
        self.preferences = preferences


def analyze_stakeholder_impact(
    solver: ValueAlignedPCenterSolver,
    stakeholders: List[Stakeholder],
) -> Dict[str, float]:
    """Compute illustrative scores from the stated proxy definitions."""
    comparison = solver.compare_solutions()
    value_solution = comparison["value_aligned_solution"]
    trade_offs = comparison["trade_offs"]

    if not comparison["trade_offs"]["is_feasible"]:
        raise ValueError("Stakeholder scores require a feasible value-aligned solution.")

    scores = {}
    for stakeholder in stakeholders:
        score = 0.0
        if "efficiency" in stakeholder.preferences:
            maximum_distance = value_solution["maximum_distance"]
            efficiency_score = 1.0 / (1.0 + maximum_distance)
            score += stakeholder.preferences["efficiency"] * efficiency_score
        if "equity" in stakeholder.preferences:
            ratio = value_solution["equity_metrics"]["equity_ratio"]
            equity_score = 0.0 if isinf(ratio) else 1.0 / ratio
            score += stakeholder.preferences["equity"] * equity_score
        if "cost" in stakeholder.preferences:
            change = trade_offs["efficiency_cost_percent"]
            cost_score = 1.0 / (1.0 + abs(change))
            score += stakeholder.preferences["cost"] * cost_score
        scores[stakeholder.name] = score * stakeholder.weight
    return scores


stakeholders = [
    Stakeholder(
        "City Government", 0.30,
        {"efficiency": 0.4, "equity": 0.4, "cost": 0.2},
    ),
    Stakeholder(
        "Community Advocates", 0.25,
        {"efficiency": 0.1, "equity": 0.7, "cost": 0.2},
    ),
    Stakeholder(
        "Fire Department", 0.25,
        {"efficiency": 0.6, "equity": 0.2, "cost": 0.2},
    ),
    Stakeholder(
        "Taxpayers Association", 0.20,
        {"efficiency": 0.3, "equity": 0.2, "cost": 0.5},
    ),
]

if trade_offs["is_feasible"]:
    stakeholder_scores = analyze_stakeholder_impact(solver, stakeholders)
    for name, score in stakeholder_scores.items():
        print(f"- {name}: illustrative score={score:.4f}")
    print(f"Sum of illustrative scores: {sum(stakeholder_scores.values()):.4f}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ax1, ax2, ax3, ax4 = axes.flat

    names = list(stakeholder_scores)
    scores = list(stakeholder_scores.values())
    bars = ax1.bar(names, scores, color=["tab:blue", "tab:green", "tab:orange", "tab:red"])
    ax1.set_title("Illustrative Stakeholder Scores")
    ax1.set_ylabel("Score (arbitrary units)")
    ax1.tick_params(axis="x", rotation=25)
    ax1.grid(True, alpha=0.3, axis="y")
    for bar, score in zip(bars, scores):
        ax1.annotate(f"{score:.3f}",
                     (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     xytext=(0, 3), textcoords="offset points", ha="center")

    categories = ["Efficiency", "Equity", "Cost"]
    bottoms = np.zeros(len(categories))
    for stakeholder in stakeholders:
        values = [stakeholder.preferences.get(category.lower(), 0) for category in categories]
        ax2.bar(categories, values, bottom=bottoms, label=stakeholder.name, alpha=0.8)
        bottoms += values
    ax2.set_title("Specified Preference Weights")
    ax2.set_ylabel("Preference weight")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3, axis="y")

    for index, stakeholder in enumerate(stakeholders):
        ax3.scatter(stakeholder.weight, stakeholder_scores[stakeholder.name], s=80)
        ax3.annotate(stakeholder.name,
                     (stakeholder.weight, stakeholder_scores[stakeholder.name]),
                     xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax3.set_title("Stakeholder Weight and Illustrative Score")
    ax3.set_xlabel("Stakeholder weight")
    ax3.set_ylabel("Illustrative score")
    ax3.grid(True, alpha=0.3)

    preference_scenarios = [
        {"name": "Current", "equity_weight": 0.6},
        {"name": "Equity-focused", "equity_weight": 0.8},
        {"name": "Efficiency-focused", "equity_weight": 0.4},
        {"name": "Balanced", "equity_weight": 0.5},
    ]
    scenario_results = []
    for scenario in preference_scenarios:
        test_constraints = EthicalConstraints(
            beta=0.3,
            gamma=0.5,
            equity_weight=scenario["equity_weight"],
            transparency_threshold=0.8,
        )
        test_comparison = ValueAlignedPCenterSolver(
            instance, test_constraints
        ).compare_solutions()
        scenario_value = test_comparison["value_aligned_solution"]
        scenario_results.append({
            "scenario": scenario["name"],
            "equity_ratio": (
                scenario_value["equity_metrics"]["equity_ratio"]
                if scenario_value["equity_metrics"] is not None else np.nan
            ),
            "maximum_distance_change_percent":
                test_comparison["trade_offs"]["efficiency_cost_percent"],
        })

    scenario_df = pd.DataFrame(scenario_results)
    positions = np.arange(len(scenario_df))
    width = 0.36
    ax4_twin = ax4.twinx()
    ax4.bar(
        positions - width / 2, scenario_df["equity_ratio"], width,
        label="Equity ratio", color="tab:green", alpha=0.75,
    )
    ax4_twin.bar(
        positions + width / 2,
        scenario_df["maximum_distance_change_percent"], width,
        label="Maximum-distance change (%)", color="tab:red", alpha=0.75,
    )
    ax4.set_title("Metrics Across Equity-Weight Scenarios")
    ax4.set_xlabel("Scenario")
    ax4.set_ylabel("Equity ratio", color="tab:green")
    ax4_twin.set_ylabel("Maximum-distance change (%)", color="tab:red")
    ax4.set_xticks(positions, scenario_df["scenario"], rotation=20)
    handles1, labels1 = ax4.get_legend_handles_labels()
    handles2, labels2 = ax4_twin.get_legend_handles_labels()
    ax4.legend(handles1 + handles2, labels1 + labels2, loc="best", fontsize=8)
    ax4.grid(True, alpha=0.3, axis="y")

    fig.tight_layout()
    plt.show()

    print(
        "Preference scenarios change the weighted objective. In this instance, "
        "all candidates are selected, so changing the objective weight cannot "
        "change the facility set."
    )
else:
    print("Illustrative stakeholder scoring requires a feasible value-aligned solution.")

## 6. Illustrative long-term monitoring scenario

The following calculation applies an assumed annual demand multiplier and budget factor to the value-aligned maximum distance, and an assumed annual adjustment to the equity ratio. These are scenario assumptions, not data-driven forecasts or consequences of the facility-location model. The compliance score is a bounded summary based on stated distance and equity thresholds; it is not a legal compliance determination. The model's facility locations are not re-optimized as conditions change.

In [ ]:
class ComplianceMonitor:
    """Calculate an illustrative monitoring series from explicit assumptions."""

    def __init__(self, solver: ValueAlignedPCenterSolver):
        self.solver = solver
        self.compliance_history = []

    def simulate_performance_over_time(self, years: int = 10) -> List[Dict]:
        """Apply assumed annual multipliers without re-optimizing facility locations."""
        if years < 1:
            raise ValueError("years must be at least 1.")

        comparison = self.solver.compare_solutions()
        value_solution = comparison["value_aligned_solution"]
        if not comparison["trade_offs"]["is_feasible"]:
            raise ValueError("Monitoring requires a feasible value-aligned solution.")

        annual_data = []
        base_distance = value_solution["maximum_distance"]
        base_equity_ratio = value_solution["equity_metrics"]["equity_ratio"]

        for year_index in range(years):
            demand_growth = 1 + 0.02 * year_index
            budget_factor = 1 - 0.01 * year_index
            if budget_factor <= 0:
                raise ValueError("The assumed budget factor must remain positive.")

            adjusted_distance = base_distance * demand_growth / budget_factor
            equity_adjustment = 1 - 0.05 * year_index
            adjusted_equity_ratio = base_equity_ratio * equity_adjustment

            distance_compliance = np.clip(1 - (adjusted_distance - 10) / 10, 0, 1)
            equity_compliance = np.clip(1 - (adjusted_equity_ratio - 1.5) / 1.5, 0, 1)
            overall_compliance = float((distance_compliance + equity_compliance) / 2)

            annual_data.append({
                "year": year_index + 1,
                "maximum_distance": adjusted_distance,
                "equity_ratio": adjusted_equity_ratio,
                "compliance_score": overall_compliance,
                "demand_multiplier": demand_growth,
                "budget_factor": budget_factor,
            })

        self.compliance_history = annual_data
        return annual_data

    def generate_compliance_report(self, annual_data: List[Dict]) -> List[Dict]:
        """Summarize the illustrative scenario and its endpoint changes."""
        if not annual_data:
            raise ValueError("annual_data must contain at least one year.")

        average_distance = float(np.mean([item["maximum_distance"] for item in annual_data]))
        average_equity_ratio = float(np.mean([item["equity_ratio"] for item in annual_data]))
        average_score = float(np.mean([item["compliance_score"] for item in annual_data]))
        first_year = annual_data[0]
        last_year = annual_data[-1]

        def percent_change(first: float, last: float) -> float:
            if isclose(first, 0.0, abs_tol=1e-12):
                return 0.0 if isclose(last, 0.0, abs_tol=1e-12) else float("inf")
            return (last - first) / first * 100

        distance_change = percent_change(
            first_year["maximum_distance"], last_year["maximum_distance"]
        )
        equity_change = percent_change(
            first_year["equity_ratio"], last_year["equity_ratio"]
        )
        score_change = percent_change(
            first_year["compliance_score"], last_year["compliance_score"]
        )

        print("Illustrative Monitoring Summary")
        print(f"- Scenario duration: {len(annual_data)} years")
        print(f"- Mean maximum distance: {average_distance:.3f} km")
        print(f"- Mean equity ratio: {average_equity_ratio:.3f}")
        print(f"- Mean bounded scenario score: {average_score:.1%}")
        print(f"- Maximum-distance change from first to last year: {distance_change:+.2f}%")
        print(f"- Equity-ratio change from first to last year: {equity_change:+.2f}%")
        print(f"- Scenario-score change from first to last year: {score_change:+.2f}%")
        print("- These values are scenario outputs, not observed performance or forecasts.")
        return annual_data


if trade_offs["is_feasible"]:
    monitor = ComplianceMonitor(solver)
    annual_data = monitor.simulate_performance_over_time(years=10)
    compliance_report = monitor.generate_compliance_report(annual_data)

    years = [item["year"] for item in annual_data]
    maximum_distances = [item["maximum_distance"] for item in annual_data]
    equity_ratios = [item["equity_ratio"] for item in annual_data]
    compliance_scores = [item["compliance_score"] for item in annual_data]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ax1, ax2, ax3, ax4 = axes.flat

    ax1.plot(years, maximum_distances, "o-", color="tab:blue", label="Scenario distance")
    ax1.axhline(10, color="tab:red", linestyle="--", label="Illustrative threshold (10 km)")
    ax1.set_title("Assumed Maximum-Distance Scenario")
    ax1.set_xlabel("Year")
    ax1.set_ylabel("Maximum distance (km)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(years, equity_ratios, "s-", color="tab:green", label="Scenario equity ratio")
    ax2.axhline(1.5, color="tab:orange", linestyle="--", label="Illustrative threshold (1.5)")
    ax2.set_title("Assumed Equity-Ratio Scenario")
    ax2.set_xlabel("Year")
    ax2.set_ylabel("Equity ratio")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3.plot(years, compliance_scores, "^-", color="tab:red", label="Scenario score")
    ax3.axhline(0.8, color="tab:gray", linestyle="--", label="Illustrative reference (80%)")
    ax3.set_title("Bounded Scenario Score")
    ax3.set_xlabel("Year")
    ax3.set_ylabel("Score (0–1)")
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    distance_range = max(maximum_distances) - min(maximum_distances)
    equity_range = max(equity_ratios) - min(equity_ratios)
    normalized_distance = (
        [1.0] * len(maximum_distances)
        if isclose(distance_range, 0.0, abs_tol=1e-12)
        else [1 - (value - min(maximum_distances)) / distance_range
              for value in maximum_distances]
    )
    normalized_equity = (
        [1.0] * len(equity_ratios)
        if isclose(equity_range, 0.0, abs_tol=1e-12)
        else [1 - (value - min(equity_ratios)) / equity_range for value in equity_ratios]
    )
    ax4.plot(years, normalized_distance, "o-", label="Relative distance performance")
    ax4.plot(years, normalized_equity, "s-", label="Relative equity-ratio performance")
    ax4.plot(years, compliance_scores, "^-", label="Bounded scenario score")
    ax4.set_title("Scenario Metrics on a 0–1 Scale")
    ax4.set_xlabel("Year")
    ax4.set_ylabel("Scaled value")
    ax4.set_ylim(0, 1)
    ax4.legend(fontsize=8)
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print("The monitoring scenario requires a feasible value-aligned solution.")

## 7. Scope, strengths, and limitations

### What this framework contributes

- It expresses group-level service equity through explicit, inspectable quantities and constraints.
- It combines the standard p-center maximum distance with a weighted group-distance measure.
- It reports group metrics and constraint violations so that the model's decisions can be examined.
- It makes parameter sensitivity visible for the specified instance.

### Important limitations

- This small example has four candidate facilities and selects all four. It therefore cannot show a choice among alternative facility sets, and its parameter analyses cannot demonstrate location changes.
- The weighted objective and equity constraint reflect the chosen mathematical definitions and input weights. Different policy choices or data would yield different assessments.
- Euclidean distance is a geometric proxy. Actual emergency response times depend on road networks, traffic, dispatch, and other operational conditions not represented here.
- The stakeholder scoring and monitoring sections use illustrative assumptions; neither supplies empirical evidence, public consultation, legal review, or a forecast.
- Exhaustive enumeration is exact for the stated small instance but grows combinatorially as the number of candidate facilities increases.

### Suitable contexts

The framework can serve as a transparent teaching example or as a starting point for carefully specified public-service location models. Any practical application requires validated data, stakeholder engagement, domain-specific constraints, and an assessment of whether the chosen equity measures are appropriate.